In [1]:
from tensorflow import keras
from kerastuner.tuners import Hyperband
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

C:\Users\ruben\AppData\Local\Temp\ipykernel_22828\1684055736.py:2: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import Hyperband


In [2]:
# --- 1. Data voorbereiden ---
X, y = make_regression(n_samples=500, n_features=10, noise=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

norm_layer = keras.layers.Normalization()
norm_layer.adapt(X_train)

In [ ]:
# --- 2. Modelbuilder functie ---
def build_model(hp):
    norm_layer = keras.layers.Normalization()
    model = keras.Sequential()
    model.add(keras.Input(shape=X_train.shape[1:]))
    model.add(norm_layer)

    # Aantal verborgen lagen: 1-3
    for i in range(hp.Int('num_layers', 1, 10)):
        model.add(
            keras.layers.Dense(
                units=hp.Int(f'units_{i}', min_value=16, max_value=256, step=16),
                activation=hp.Choice('activation', ['relu', 'tanh'])
            )
        )
    
    model.add(keras.layers.Dense(1))  # outputlaag voor regressie

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=hp.Float('learning_rate', 1e-5, 1e-2, sampling='log')
        ),
        loss='mse',
        metrics=['RootMeanSquaredError']
    )
    
    return model

In [ ]:
# --- 3. Bayesian Optimization Tuner ---
tuner = Hyperband(
    build_model,
    objective='val_mse',
    max_epochs=50,  # maximale epochs per trial
    factor=2,       # bepaalt hoeveel trials worden doorgelicht bij “early stopping”
    directory='hyperband_opt_dir',
    project_name='mlp_regression'
)

In [ ]:
# --- 4. Start hyperparameter tuning ---
early_stopping_cb = keras.callbacks.EarlyStopping(
    monitor='val_mae',       # was val_root_mean_squared_error
    patience=10,
    min_delta=0.001,         # kleine verbetering in MAE is al goed
    restore_best_weights=True,
    mode='min'               # MAE moet omlaag
)

tuner.search(
    X_train, y_train,
    epochs=50,
    validation_data=(X_val, y_val),
    batch_size=32,
    callbacks=[early_stopping_cb],
    verbose=1
)

Trial 90 Complete [00h 00m 08s]
val_mse: 156.52052307128906

Best val_mse So Far: 7.360587120056152
Total elapsed time: 00h 03m 24s


In [12]:
# --- 5. Beste hyperparameters en model ---
best_hp = tuner.get_best_hyperparameters(1)[0]
print("Beste hyperparameters:")
print(best_hp.values)

best_model = tuner.get_best_models(1)[0]

Beste hyperparameters:
{'num_layers': 3, 'units_0': 128, 'activation': 'relu', 'learning_rate': 0.0076208407593745765, 'units_1': 112, 'units_2': 64, 'tuner/epochs': 50, 'tuner/initial_epoch': 17, 'tuner/bracket': 3, 'tuner/round': 3, 'tuner/trial_id': '0046'}


In [13]:
best_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 normalization (Normalizati  (None, 10)                21        
 on)                                                             
                                                                 
 dense (Dense)               (None, 128)               1408      
                                                                 
 dense_1 (Dense)             (None, 112)               14448     
                                                                 
 dense_2 (Dense)             (None, 64)                7232      
                                                                 
 dense_3 (Dense)             (None, 1)                 65        
                                                                 
Total params: 23174 (90.53 KB)
Trainable params: 23153 (90.44 KB)
Non-trainable params: 21 (88.00 Byte)
__________________

In [ ]:
best_model.evaluate(X_test, y_test)

In [ ]:
X_new = X_test[:3]  # pretend these are new instances
y_proba = best_model.predict(X_new)